# 01 - Extraction des caracteristiques

Dans ce notebook, je transforme chaque image de mon corpus en un vecteur de caracteristiques, celui
qu'utilise la steganalyse classique. Ce sont ces vecteurs qui serviront ensuite a entrainer et tester
mes detecteurs.

L'extraction SRM est lente et mon corpus est gros, alors je la parallelise sur tous les coeurs et je
mets chaque resultat en cache. Ainsi, si ma session se coupe, le calcul reprend la ou il s'etait arrete.

Je centralise toute la configuration dans la premiere cellule : je change une valeur, je ne touche a
rien d'autre.

## Configuration

C'est la seule cellule que je modifie selon ce que je veux extraire. Par exemple SPAM rapide sur tout,
ou SRMQ1 seulement sur les algorithmes adaptatifs.

In [ ]:
# ================= CONFIGURATION =================
FEATURE  = 'srmq1'                 # 'spam', 'srmq1' ou 'srm'
SOURCES  = ['natural', 'sd', 'sdxl', 'adm']
ALGOS    = ['uniward', 'hill']     # algorithmes a extraire ; [] pour covers seuls
PAYLOADS = [0.4]                   # charges utiles a extraire
INCLURE_COVER = True               # extraire aussi les images vierges
N_WORKERS = 0                      # 0 = tous les coeurs disponibles
SEED = 42
# ================================================
print('Configuration :', FEATURE, '| sources', SOURCES, '| algos', ALGOS, '| payloads', PAYLOADS)

## Environnement et corpus

Je detecte si je suis sur Colab ou sur un VPS et j'adapte les chemins. Sur VPS, je place l'archive de
mon corpus dans le dossier memoire_data a cote du notebook, et je la laisse se decompresser toute seule.

In [ ]:
import os, glob, shutil, time
import numpy as np
np.random.seed(SEED)

# Je choisis les chemins selon l'environnement
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/memoire_data'
    ROOT = '/content/corpus'
except Exception:
    DATA_DIR = os.path.abspath('./memoire_data')   # VPS ou local
    ROOT = os.path.abspath('./corpus')
os.makedirs(DATA_DIR, exist_ok=True)

FEAT_DIR = f'{DATA_DIR}/features_{FEATURE}'
os.makedirs(FEAT_DIR, exist_ok=True)

# Si le corpus n'est pas la, je le restaure depuis ma derniere archive
if not os.path.isdir(f'{ROOT}/natural/cover'):
    zips = sorted(glob.glob(f'{DATA_DIR}/corpus_*.zip'))
    if zips:
        print('Je decompresse le corpus depuis', zips[-1])
        shutil.unpack_archive(zips[-1], ROOT)
    else:
        print('Aucun corpus trouve. Je dois placer une archive corpus_*.zip dans', DATA_DIR)

N_WORKERS = N_WORKERS or (os.cpu_count() or 1)
print('Coeurs que j\'utilise :', N_WORKERS, '| cache :', FEAT_DIR)

## Installation

In [ ]:
# sealwatch me fournit les extracteurs de caracteristiques, le reste sert a lire les images
!pip install -q sealwatch imageio scipy
print('Installation terminee.')

## Fonction d'extraction

L'extracteur attend un tableau, pas un chemin, donc je charge l'image d'abord. Je fais un test sur une
seule image pour connaitre la dimension et le temps par image, ce qui me permet d'estimer la duree totale.

In [ ]:
import imageio.v2 as imageio
import sealwatch as sw

def load_gray(path):
    # Je charge l'image en niveaux de gris
    x = np.asarray(imageio.imread(path))
    if x.ndim == 3:
        x = x[..., 0]
    return x

def to_vec(f):
    # sealwatch me rend parfois un dictionnaire de sous modeles, je l'aplatis en un vecteur
    if isinstance(f, dict):
        return np.asarray(sw.tools.flatten(f), dtype=np.float32).ravel()
    return np.asarray(f, dtype=np.float32).ravel()

def _extracteur():
    # Je choisis l'extracteur selon ma configuration
    if FEATURE == 'spam':
        return sw.spam
    if FEATURE == 'srmq1':
        return sw.srmq1
    return sw.srm

def extract(path):
    return to_vec(_extracteur().extract(load_gray(path)))

un = sorted(glob.glob(f'{ROOT}/natural/cover/*.pgm'))[:1]
if un:
    t = time.time(); d = extract(un[0]).shape[0]
    print(f'dimension {d}, temps par image {time.time()-t:.2f}s (avant parallelisation)')

## Extraction parallele avec cache reprenable

Je traite les images par blocs, en parallele sur tous mes coeurs, et je sauvegarde apres chaque bloc.
Comme ca, une coupure ne me fait perdre au plus qu'un bloc. Je peux relancer la cellule autant que je veux.

In [ ]:
from concurrent.futures import ProcessPoolExecutor

def extract_set(paths, cache_file, bloc=200):
    # Si j'ai deja ce cache, je le recharge sans recalculer
    if os.path.exists(cache_file):
        return np.load(cache_file)
    tmp = cache_file + '.part.npy'
    feats = list(np.load(tmp)) if os.path.exists(tmp) else []
    if feats:
        print('   je reprends a', len(feats))
    with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
        i = len(feats)
        while i < len(paths):
            lot = paths[i:i + bloc]
            feats.extend(ex.map(extract, lot, chunksize=4))
            i += len(lot)
            np.save(tmp, np.array(feats, dtype=np.float32))   # mon point de reprise
            print('   ', i, '/', len(paths))
    arr = np.array(feats, dtype=np.float32)
    np.save(cache_file, arr)
    if os.path.exists(tmp):
        os.remove(tmp)
    return arr

def paths_of(source, setname):
    # setname vaut 'cover' ou '<algo>_p<payload>'
    if setname == 'cover':
        return sorted(glob.glob(f'{ROOT}/{source}/cover/*.pgm'))
    algo, p = setname.split('_p')
    return sorted(glob.glob(f'{ROOT}/{source}/{algo}/*_p{p}.pgm'))

# Je construis la liste des ensembles a extraire depuis ma configuration
setnames = (['cover'] if INCLURE_COVER else []) + [f'{a}_p{p}' for a in ALGOS for p in PAYLOADS]
print('Ensembles que je vise :', setnames, '\n')

for source in SOURCES:
    for setname in setnames:
        paths = paths_of(source, setname)
        if not paths:
            print(f'{source} {setname}: aucune image, je passe'); continue
        cache_file = f'{FEAT_DIR}/{source}__{setname}.npy'
        if os.path.exists(cache_file):
            print(f'{source} {setname}: deja en cache'); continue
        t = time.time()
        print(f'{source} {setname}: {len(paths)} images')
        extract_set(paths, cache_file)
        print(f'   termine en {(time.time()-t)/60:.1f} min')
print('\nJ\'ai fini l\'extraction, ou je l\'ai reprise.')

## Verification

Je liste mes caches et je verifie leurs dimensions, pour m'assurer que tout est coherent avant les experiences.

In [ ]:
caches = sorted(glob.glob(f'{FEAT_DIR}/*.npy'))
print(f'{len(caches)} fichiers dans {FEAT_DIR}\n')
for c in caches:
    a = np.load(c, mmap_mode='r')
    print(f'  {os.path.basename(c):24s} {a.shape}')
print(f'\nEnsembles que je visais cette fois : {len(SOURCES) * len(setnames)}')

## Suite

Je recupere le dossier des caracteristiques, sur Drive en Colab ou par scp depuis mon VPS, et je pointe
le notebook 02 vers le meme FEATURE. Je note le temps par image et la duree totale dans mon journal.

Pour lancer sur un VPS sans interface, je suis mon guide VPS_SETUP.md a la racine du depot.